### Mapa interactivo

In [0]:
%pip install folium

In [0]:
import folium

CENTROIDES = {
 "AC":(-9.0,-70.5),"AL":(-9.6,-36.8),"AP":(0.9,-52.0),"AM":(-3.4,-64.7),
 "BA":(-12.6,-41.7),"CE":(-5.5,-39.3),"DF":(-15.8,-47.9),"ES":(-19.2,-40.0),
 "GO":(-15.8,-49.8),"MA":(-4.6,-45.2),"MT":(-12.6,-56.0),"MS":(-20.5,-54.6),
 "MG":(-18.5,-44.0),"PA":(-3.8,-52.5),"PB":(-7.2,-36.7),"PR":(-25.3,-52.1),
 "PE":(-8.4,-37.9),"PI":(-7.7,-42.7),"RJ":(-22.9,-43.2),"RN":(-5.4,-36.9),
 "RS":(-30.0,-53.2),"RO":(-11.5,-63.6),"RR":(2.7,-62.1),"SC":(-27.3,-50.5),
 "SP":(-22.2,-48.8),"SE":(-10.7,-37.4),"TO":(-10.2,-48.3)}

#  Nombres completos de los estados
NOMBRES = {
 "AC":"Acre","AL":"Alagoas","AP":"Amapá","AM":"Amazonas","BA":"Bahía",
 "CE":"Ceará","DF":"Distrito Federal","ES":"Espírito Santo","GO":"Goiás",
 "MA":"Maranhão","MT":"Mato Grosso","MS":"Mato Grosso do Sul","MG":"Minas Gerais",
 "PA":"Pará","PB":"Paraíba","PR":"Paraná","PE":"Pernambuco","PI":"Piauí",
 "RJ":"Río de Janeiro","RN":"Río Grande del Norte","RS":"Río Grande del Sur",
 "RO":"Rondônia","RR":"Roraima","SC":"Santa Catarina","SP":"São Paulo",
 "SE":"Sergipe","TO":"Tocantins"}

geo = spark.sql("""
  SELECT cliente_estado AS uf,
         SUM(pedidos)                               AS pedidos,
         SUM(tasa_retraso_pct*pedidos)/SUM(pedidos) AS tasa
  FROM big_data_2026.olist.gold_corredores_envio
  GROUP BY cliente_estado
""").toPandas()

geo["lat"] = geo.uf.map(lambda u: CENTROIDES[u][0])
geo["lon"] = geo.uf.map(lambda u: CENTROIDES[u][1])

max_pedidos = geo.pedidos.max()

m = folium.Map(location=[-14, -52], zoom_start=4)
for _, r in geo.iterrows():
    folium.CircleMarker(
        (r.lat, r.lon),
        radius=6 + (r.pedidos / max_pedidos) * 20,
        color="red" if r.tasa > 10 else "orange" if r.tasa > 6 else "green",
        fill=True, fill_opacity=0.6,
        #  Nombre completo en lugar de iniciales
        tooltip=f"{NOMBRES.get(r.uf, r.uf)}: {r.tasa:.1f}% | {int(r.pedidos)} pedidos"
    ).add_to(m)
import requests   # ← agrégalo arriba con los otros imports

#  Contorno de Brasil con línea gruesa (se dibuja DEBAJO de las burbujas)
try:
    url = "https://raw.githubusercontent.com/johan/world.geo.json/master/countries/BRA.geo.json"
    bra = requests.get(url, timeout=15).json()
    folium.GeoJson(
        bra,
        style_function=lambda f: {
            "color": "#222222",   # gris casi negro
            "weight": 4,          # ← línea gruesa
            "opacity": 0.9,
            "fill": False,        # sin relleno para no tapar las burbujas
        },
    ).add_to(m)
except Exception as e:
    print("No se pudo cargar el contorno de Brasil:", e)

displayHTML(m._repr_html_())